## This is the starting dataset for the infleqtion project. 
---
The idea here is to create the circuits first using cirq to make sure they work as expected, then three json entries would be made per circuit noting the prompting style that should be used to create the desired circuit. 

#### Prompting Styles 
- [[Technical, using precise quantum notation of desired circuit, ex. "Prepare |phi+> on two qubits" ],
    [Conversational, using everday language, ex. "Entangle 2 qubits" ],
    [Narrtive, describes the outcome of a circuit, ex. "Generate a circuit that produce 00 and 11 with equal probabilities]]

In [1]:
#bell state circuit
import cirq

bell_circuit = cirq.Circuit()
q0, q1 = cirq.LineQubit.range(2)
bell_circuit.append(cirq.H(q0)) 
bell_circuit.append(cirq.CNOT(q0, q1)) 
bell_circuit
print(cirq.final_state_vector(bell_circuit))


[0.70710677+0.j 0.        +0.j 0.        +0.j 0.70710677+0.j]


In [2]:
h_circuit = cirq.Circuit()

q0 = cirq.LineQubit.range(1)
h_circuit.append(cirq.H(q0[0]))
print(h_circuit)


0: ───H───


In [3]:

#Syntactic Check -- Makes sure that everything actually runs 
def check_syntax(llm_code: str) -> dict:
    try:
        exec_globals = {}
        exec(llm_code, exec_globals)
        return {"passed": True, "error": None}
    except Exception as e:
        return {"passed": False, "error": str(e)}

In [2]:
#Structural Check -- does it have the right gates and qubits
def check_structure(llm_circuit: cirq.Circuit, control_circuit: cirq.Circuit,) -> dict:
    results = {} 

    # Check amount of qubits 
    llm_qubits = len(llm_circuit.all_qubits())
    control_qubits = len(control_circuit.all_qubits())
    results["qubit_count_match"] = llm_qubits == control_qubits 

    # Check gate type present 
    def get_gates(circuit):
        return set(
            type(op.gate).__name__ #Collects names of gates as a set
            for moment in circuit
            for op in moment.operations
            if op.gate is not None
        )

    llm_gates = get_gates(llm_circuit)
    control_gates = get_gates(control_circuit)
    results["gate_types_match"] = control_gates.issubset(llm_gates)
    results["llm_gates"] = list(llm_gates) 
    results["control_gates"] = list(control_gates)

    #Measurement Check

    llm_has_measurement = any(

        isinstance(op.gate, cirq.MeasurementGate)

        for moment in llm_circuit
        for op in moment.operations
    )

    control_has_measurement = any(
        isinstance(op.gate, cirq.MeasurementGate)
        for moment in control_circuit
        for op in moment.operations
    )
    results["measurement_match"] = llm_has_measurement == control_has_measurement

    # results["circuit_depth_llm"] = len(llm_circuit)
    # results["circuit_depth_control"] = len(control_circuit)
    results["depth_within_tolerance"] = len(llm_circuit) <= len(control_circuit) * 2 

    return results

In [3]:
# Semantic Check - checks the output distribution
import numpy as np
def check_semantics(llm_circuit: cirq.Circuit, control_circuit: cirq.Circuit, reps: int = 1000, tolerance: float = 0.1) -> dict:
    simulator = cirq.Simulator()
    
    #MESUREMENT FUNCTION '
    def ensure_measured(circuit):
        qubits = sorted(circuit.all_qubits())
        if not any(isinstance(op.gate, cirq.MeasurementGate)
                   for moment in circuit for op in moment.operations):
            circuit = circuit + cirq.measure(*qubits, key='result') #* unpacks list of qubits as their own arguments 
            #key+'result' makes a name for measurements to call

        return circuit

    llm_measured = ensure_measured(llm_circuit.copy())  
    control_measured = ensure_measured(control_circuit.copy())
    
    llm_result = simulator.run(llm_measured, repetitions=reps)
    control_result = simulator.run(control_measured, repetitions=reps)

    #Build Probability(Result) distributions
    def get_distribution(result, reps):
        counts = result.multi_measurement_histogram(keys=['result']) # counts the number of outcomes 
        return {k: v/reps for k, v in counts.items()}  #turns number into probability
    llm_dist = get_distribution(llm_result, reps)
    control_dist = get_distribution(control_result, reps)

    #Compare distributions 
    all_keys = set(llm_dist.keys()) | set(control_dist.keys())
    
    distribution_match = all(
        abs(llm_dist.get(k,0) - control_dist.get(k,0)) <= tolerance
        for k in all_keys 
    )

    # STATE VECTOR
    def get_state_vector(circuit):
        no_measurement = cirq.drop_terminal_measurements(circuit)
        return cirq.final_state_vector(no_measurement)

    llm_vector = get_state_vector(llm_circuit)
    control_vector = get_state_vector(control_circuit)

    #comparing the vectors
    fidelity = float(abs(np.dot(np.conj(llm_vector), control_vector))**2)
    assert cirq.equal_up_to_global_phase(llm_vector, control_vector)
    # np.dot(np.conj()creates inner products of the two vectors 
    # the absolute value and square on it makes the quantum fidelity 
    # quantum fidelity of 1 means that the two vectors are the same and 0 means that they are orthagonal or completely different. 
    
    # RETURN combined result
    return {
        "passed": distribution_match, 
        # "passed": fidelity >= 0.9,
        "fidelity_score": fidelity,
        # "llm_distribution": {str(k): v for k, v in llm_dist.items()},
        # "control_distribution": {str(k): v for k, v in control_dist.items()}
    }

    
   



    

       

In [4]:
# THE BIG ONE 

def validate_circuit(llm_code: str, control_circuit: cirq.Circuit) -> dict:
    report = {}
    # Step 1: Syntax
    syntax = check_syntax(llm_code)
    report["syntax"] = syntax

    if not syntax["passed"]:
        report["verdict"] = "invalid_code"
        return report
    
    #Step 2: Extracting da circuit
    exec_globals = {}
    exec(llm_code, exec_globals)
    llm_circuit = next(
        (v for v in exec_globals.values() if isinstance(v, cirq.Circuit)),
        None
    )
    if llm_circuit is None:
        report["verdict"] = "no_circuit_found"
        return report
    
    # Step 3: Structure
    structure = check_structure(llm_circuit, control_circuit)
    report["structure"] = structure

    # Step 4: Semantics
    semantics = check_semantics(llm_circuit, control_circuit)
    report["semantics"] = semantics 

    #Step 5: FINAL Outcome 
    if not structure["qubit_count_match"]:
        report["verdict"] = "missing_qubits"
    elif not structure["gate_types_match"]:
        report["verdict"] = "wrong_gate"
    elif not structure["measurement_match"]:
        report["verdict"] = "missing_measurement"
    elif not semantics["passed"]:
        report["verdict"] = "semantic_fail"
    else:
        report["verdict"] = "success"

    return report


In [5]:
bell_test_code = """import cirq

# Create a quantum circuit with 2 qubits\nqubits = cirq.LineQubit.range(2)
circuit = cirq.Circuit()
circuit.append([
    cirq.H(qubits[0]),
    cirq.CNOT(qubits[0], qubits[1])
])
"""
validate_circuit(bell_test_code, bell_circuit)

NameError: name 'bell_circuit' is not defined

In [8]:
bell_test2_code = """import cirq\n\n# Create a quantum circuit with 2 qubits\nqubits = cirq.LineQubit.range(2)  # qubits 0 and 1\ncircuit = cirq.Circuit()\n\n# Add gates to create the Bell state |Φ⁺⟩ = (|00⟩ + |11⟩)/√2\ncircuit.append([\n    # Apply Hadamard gate to the first qubit\n    cirq.H(qubits[0]),\n\n    # Apply CNOT gate with first qubit as control and second as target\n    cirq.CNOT(qubits[0], qubits[1])\n])\n\n# Print the circuit\nprint("Circuit to create Bell state |Φ⁺⟩:")\nprint(circuit)\n\n# Visualize the circuit\nprint("\\nCircuit diagram:")\nprint(circuit.to_text_diagram())\n"""

validate_circuit(bell_test2_code, bell_circuit)

Circuit to create Bell state |Φ⁺⟩:
0: ───H───@───
          │
1: ───────X───

Circuit diagram:
0: ───H───@───
          │
1: ───────X───
Circuit to create Bell state |Φ⁺⟩:
0: ───H───@───
          │
1: ───────X───

Circuit diagram:
0: ───H───@───
          │
1: ───────X───


{'syntax': {'passed': True, 'error': None},
 'structure': {'qubit_count_match': True,
  'gate_types_match': True,
  'llm_gates': ['HPowGate', 'CXPowGate'],
  'control_gates': ['HPowGate', 'CXPowGate'],
  'measurement_match': True,
  'circuit_depth_llm': 2,
  'circuit_depth_control': 2,
  'depth_within_tolerance': True},
 'semantics': {'passed': True,
  'fidelity_score': 0.9999998807907104,
  'llm_distribution': {'(3,)': 0.493, '(0,)': 0.507},
  'control_distribution': {'(0,)': 0.523, '(3,)': 0.477}},
 'verdict': 'success'}

In [ ]:

# JSON Database for Notion Work
notion_database = {
  "database_name": "Quantum Circuit Tracker",
  "record_count": 26,
  "schema": {
    "Circuit": "object",
    "Description of Utility": "object",
    "Difficulty": "object",
    "Expected Gates": "object",
    "Status": "object"
  },
  "records":[
    {
      "Circuit": "Two-Layer Variational Circuit",
      "Description of Utility": "Parameterized ansatz with two layers of rotations and entanglers on three qubits. Used in VQE and quantum ML.",
      "Difficulty": "Advanced",
      "prompt": "Construct a two-layer parameterized ansatz circuit on three qubits in Cirq. Each layer should consist of RY rotation gates on all qubits followed by CNOT entangling gates between adjacent qubits. Use symbolic parameters for rotation angles. Return the resulting Cirq circuit.",
      "prompt_style": "technical",
      "num_qubits": 3,
      "expected_gates": ["RY, CNOT, RY, CNOT"],
      "expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability.",
      "resulted_code": "python\nimport cirq\n\n# Create a circuit with three qubits\nq0, q1, q2 = cirq.LineQubit.range(3)\ncircuit = cirq.Circuit()\n\n# Define symbolic rotation angles\ntheta = cirq různý.Symbol('θ')\nphi = cirq Routine.Symbol('φ')\n\n# First layer\ncircuit.append([\n    cirq.ry(theta).on(q0),\n    cirq.ry(theta).on(q1),\n    cirq.ry(theta).on(q2),\n    cirq.CNOT(q0, q1),\n    cirq.CNOT(q1, q2)\n])\n\n# Second layer\ncircuit.append([\n    cirq.ry(phi).on(q0),\n    cirq.ry(phi).on(q1),\n    cirq.ry(phi).on(q2),\n    cirq.CNOT(q0, q1),\n    cirq.CNOT(q1, q2)\n])\n\nprint(circuit)\n",
      "validated_verdict": "right idea, halluciated code, slop",
    }
    {
      "Circuit": "Two-Layer Variational Circuit",
      "Description of Utility": "Parameterized ansatz with two layers of rotations and entanglers on three qubits. Used in VQE and quantum ML.",
      "Difficulty": "Advanced",
      "prompt": "Generate a cirq quantum circuit that produces a parameterized circuit with symbolic rotation angles (not numerical values). The circuit structure should show two repeated layers of RY gates followed by CNOT ladders.",
      "prompt_style": "narrative",
      "num_qubits": 3,
      "expected_gates": ["RY, CNOT, RY, CNOT"],
      "expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability.",
      "resulted_code": "import cirq\nimport sympy as sp\n\n# Define symbolic rotation angles\ntheta, phi = sp.symbols('theta phi')\n\n# Create a qubit register\nq0, q1 = cirq.LineQubit.range(2)\n\n# Define a single RY gate layer\nry_layer = (cirq.Ry(theta) @ cirq.Ry(phi)) ** 2\ncircuit_step_a = ry_layer(q0)\n\n# Create a CNOT ladder layer\ncircuit_step_b = cirq.CNOT(q0, q1)\n\n# Combine the layers into a single parameterized circuit\ncircuit = cirq.FrozenTrial(\n    cirq Circuit =\n        cirqнг(circuit_step_a, circuit_step_b), \n    parameters=[theta, phi],\n    strategy=None)\n    \n# Visualization\nprint(circuit.map(lambda op: op with copy_operations=False)) \n\n# Applying the parameterized circuit with numerical angles\ntheta_value, phi_value = 0.5, 1.123  \nfinal_circuit = circuit.substitute(theta=theta_value, phi=phi_value)\n",
    "validated_verdict": "right idea, hallucinated code, visualization"
    },
    {
      "Circuit": "Two-Layer Variational Circuit",
      "Description of Utility": "Parameterized ansatz with two layers of rotations and entanglers on three qubits. Used in VQE and quantum ML.",
      "Difficulty": "Advanced",
      "prompt": "Generate a Two-Layer Variational Circuit as a cirq circuit.",
      "prompt_style": "conversational",
      "num_qubits": 3,
      "expected_gates": ["RY, CNOT, RY, CNOT"],
      "expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability."
      "resulted_code": """import cirq\n\n# Create two qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\n\n# Apply the CNOT gate to q0 and q1\ncircuit = cirq.Circuit(\n    cirq.X(q0) ** 0.5,  # Optional, applies a Hadamard gate to q0\n    cirq.CNOT(q0, q1),\n    cirq.MCTLayers()[0][cirq.MeasurementGate(1)()(q0)],  # Measure q0\n    cirq.measure(q0, key='q0'),\n    cirq.measure(q1, key='q1')\n)\n\n# Simulate the circuit and print the output\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\n\nprint(result.histogram(key='q0'))\n""",
      "validated_verdict:": "invalid code, a lot of halucinations, inproper gates,",
    }, 
    {
      "Circuit": "Phase Kickback",
      "Description of Utility": "Demonstrates how phase from a target qubit transfers back to a control qubit. Core subroutine in Shor's and Grover's algorithms.",
      "Difficulty": "Advanced",
      "Expected Gates": "H, CNOT, H",
      "Status": "Haven't Touched"
    },
    {
      "Circuit": "Bell State (|Φ+⟩)",
      "Description of Utility": "Creates a maximally entangled two-qubit state producing only |00⟩ and |11⟩ outcomes. Core benchmark for this project.",
      "Difficulty": "Beginner",
      "prompt": "Prepare the Bell state |Phi+> on two qubits as a Cirq circuit",
      "prompt_style": "technical",
      "num_qubits": 2,
      "expected_gates": ["H", "CNOT"],
      "expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability."
      "resulted code": "```python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit(\n    # Apply Hadamard to qubit0\n    cirq.H(qubit0),\n    # Apply CNOT with qubit0 as control and qubit1 as target\n    cirq.CNOT(qubit0, qubit1)\n)\n\n# Print the circuit\nprint("Circuit to prepare |Φ⁺⟩:")\nprint(circuit)\n```",
      "validated verdict": "success"
    },
    {
      "Circuit": "Bell State (|Φ+⟩)",
      "Description of Utility": "Creates a maximally entangled two-qubit state producing only |00⟩ and |11⟩ outcomes. Core benchmark for this project.",
      "Difficulty": "Beginner",
      "prompt": "Generate a quantum circuit that produces 00 and 11 with equal probibility",
      "prompt_style": "narrative",
      "num_qubits": 2,
      "expected_gates": ["H", "CNOT"],
      "expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability.",
      "resulted_code": """python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit()\ncircuit.append(cirq.H(qubit0))  # Apply Hadamard to qubit0\ncircuit.append(cirq.CNOT(qubit0, qubit1))  # Apply CNOT with qubit0 as control and qubit1 as target\n\n# Print the circuit\nprint("Circuit:")\n print(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nfinal_state = result.final_state_vector\n\nprint("\\nFinal state vector:")\nprint(final_state)\n\n# Measure the probabilities\nprint("\\nMeasurement probabilities:")\nprint(result.measurements)\n""",
    "validated_verdict": "invalid code, spacing error"
    },
    {
      "Circuit": "Bell State (|Φ+⟩)",
      "Description of Utility": "Creates a maximally entangled two-qubit state producing only |00⟩ and |11⟩ outcomes. Core benchmark for this project.",
      "Difficulty": "Beginner",
      "prompt": "Entangle 2 qubits as a Cirq Circuit",
      "prompt_style": "conversational",
      "num_qubits": 2,
      "expected_gates": ["H", "CNOT"],
      "expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability."
      "resulted_code": """python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n#Add a Hadamard gate to qubit0 to create superposition\ncircuit.append(cirq.H(qubit0))\n\n# Add a CNOT gate to entangle qubit0 and qubit1\ncircuit.append(cirq.CNOT(qubit0, qubit1))\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nprint("\\nFinal state:)\nprint(result.final_state)\n"",
      "validated_verdict:": "invalid_code, inccorectly structured print statement, hallucinated state vector code",
    },
    {
      "Circuit": "Hadamard (Single Qubit)",
      "Description of Utility": "Puts a single qubit into equal superposition of |0⟩ and |1⟩. Foundational gate used in nearly every quantum algorithm.",
      "Difficulty": "Beginner",
      "prompt": "Apply a Hadamard gate to a single qubit initialized in the |0⟩ computational basis state and return the resulting Cirq circuit.",
      "prompt_style": "technical",
      "num_qubits": 1,
      "expected_gates": ["H"],
      "expected_behavior": "After H is applied the qubit is put into superposition"
      "resulted code:```python\nimport cirq\n\n# Create a single qubit circuit\nqubit = cirq.LineQubit(0)\ncircuit = cirq.Circuit()\n\n# Add the Hadamard gate to the qubit\ncircuit.append(cirq.H(qubit))\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n```",
      "validated verdict: success"
    },
    {
      "Circuit": "Hadamard (Single Qubit)",
      "Description of Utility": "Puts a single qubit into equal superposition of |0⟩ and |1⟩. Foundational gate used in nearly every quantum algorithm.",
      "Difficulty": "Beginner",
      "prompt": "Apply a Hadamard gate to a single qubit initialized in the |0⟩ computational basis state and return the resulting Cirq circuit.",
      "prompt_style": "conversational",
      "num_qubits": 1,
      "expected_gates": ["H"],
      "expected_behavior": "After H is applied the qubit is put into superposition"
      "resulted_code": "```python\nimport cirq\n\n# Create a circuit with a single qubit initialized in the |0⟩ state\nqubit = cirq.LineQubit(0)\ncircuit = cirq.Circuit(cirq.H(qubit))\n\nprint(circuit)\n```", 
      "validated_verdict": "success", 
    },
    {
      "Circuit": "Hadamard (Single Qubit)",
      "Description of Utility": "Puts a single qubit into equal superposition of |0⟩ and |1⟩. Foundational gate used in nearly every quantum algorithm.",
      "Difficulty": "Beginner",
      "prompt": "Put a qubit in superposition and model as a Cirq Circuit.",
      "prompt_style": "narrative",
      "num_qubits": 1,
      "expected_gates": ["H"],
      "expected_behavior": "After H is applied the qubit is put into superposition"
      "resulted_code": "```python\nimport cirq\n\n# Create two qubits\nq0 = cirq.LineQubit(0)\n\n# Create a circuit containing an H gate applied to q0\ncircuit = cirq.Circuit(\n    cirq.H(q0),  # Apply Hadamard gate to q0\n)\n\nprint(circuit)  # Print the circuit\n```",
      "validated_verdict:": "success",
    },
    {
      "Circuit": "CNOT Gate",
      "Description of Utility": "Flips target qubit conditional on control qubit being |1⟩. Primary two-qubit entangling gate in most frameworks.",
      "Difficulty": "Beginner",
      "prompt": "Construct a two-qubit Cirq circuit implementing a CNOT gate with qubit 0 as the control and qubit 1 as the target. Return the resulting Cirq circuit.",
      "prompt_style": "technical",
      "num_qubits": 1,
      "expected_gates": ["CNOT"],
      "expected_behavior": "After CNOT is applied the qubits become entangled"
      "resulted code":"```python\nimport cirq\n\n# Create two qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Define the CNOT operation\ncircuit = cirq.Circuit(\n    # CNOT operation\n    cirq.CNOT(qubit0, qubit1),\n)\n\nprint(circuit)\n```",
      "validated verdict: success"
    },
    {
      "Circuit": "CNOT Gate",
      "Description of Utility": "Flips target qubit conditional on control qubit being |1⟩. Primary two-qubit entangling gate in most frameworks.",
      "Difficulty": "Beginner",
      "prompt": "Create a cirq circuit that entangles 2 qubits with q0 as a control and q1 as a target. Implement the CNOT gate.",
      "prompt_style": "conversational",
      "num_qubits": 1,
      "expected_gates": ["CNOT"],
      "expected_behavior": "After CNOT is applied the qubits become entangled"
      "resulted_code": "```python\nimport cirq\n\n# Create two qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\n\n# Create a CNOT gate\ncnot_gate = cirq.CNOT(q0, q1)\n\n# Print the CNOT gate\nprint(cnot_gate)\n\n# Create a circuit with the CNOT gate\ncircuit = cirq.Circuit(cnot_gate)\n\n# Add a measurement for the qubits\ncircuit.append([\n    cirq.measure(q0, key='q0'),\n    cirq.measure(q1, key='q1')\n])\n\n# Print the circuit\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1)\nprint(result)\n```", 
      "validated_verdict": "invalid, proper circuit, added measurement gates, ran a simulation", 
    },
    {
      "Circuit": "CNOT Gate",
      "Description of Utility": "Flips target qubit conditional on control qubit being |1⟩. Primary two-qubit entangling gate in most frameworks.",
      "Difficulty": "Beginner",
      "prompt": "Create a cirq circuit that models the computational basis state |11⟩ with 100% probability.",
      "prompt_style": "narrative",
      "num_qubits": 1,
      "expected_gates": ["CNOT"],
      "expected_behavior": "After CNOT is applied the qubits become entangled"
      "resulted_code": "```python\nimport cirq\n\n# Define two qubits to represent the computational basis state |11⟩\nq0 = cirq.GridQubit(0, 0)\nq1 = cirq.GridQubit(1, 0)\n\n# Prepare the qubits in the |1⟩ state\ncircuit = cirq.Circuit(\n    cirq.X(q0),\n    cirq.X(q1),\n    cirq.measure(q0, key='q0'),\n    cirq.measure(q1, key='q1'),\n)\n\n# Simulate the circuit to find the outcome\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)  # Run the circuit for 1000 repetitions\n\n# Print the outcome\nprint(result)\n```",
      "validated_verdict:": "invalid circuit, applied x twice, didn't print a circuit, ran a sim and printed outcomes",
    },
    {
      "Circuit": "Grover's Diffusion Operator",
      "Description of Utility": "Amplitude amplification step of Grover's search algorithm. Reflects amplitudes about their mean to boost target state probability. Advanced stretch circuit.",
      "Difficulty": "Expert",
      "Expected Gates": "H, X, H, CNOT, H, X, H",
      "Status": "Haven't Touched"
    },
    {
      "Circuit": "Ambiguous Entanglement Prompt",
      "Description of Utility": "Intentionally vague prompt like 'Make two qubits the same somehow.' Tests whether LLM asks for clarification or picks a valid interpretation.",
      "Difficulty": "Intermediate",
      "prompt": "Prepare a two-qubit state exhibiting quantum correlation. Model as a cirq circuit.",
      "prompt_style": "technical",
      "num_qubits": "2+",
      "expected_gates": ["TBD — model chooses"],
      "expected_behavior": "Model uses its own reasoning skills to create entangled qubits."
      "resulted code:```python\nimport cirq\n\n# Create two qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\n\n# Create a circuit for the Bell state\ncircuit = cirq.Circuit(\n    cirq.H(q0),\n    cirq.CNOT(q0, q1),\n    cirq.H(q0)\n)\n\nprint(circuit)\n```",
      "validated verdict: "success, didn't prepare bell-state, made uniform superposition, still exceptable"
    },
    {
      "Circuit": "Ambiguous Entanglement Prompt",
      "Description of Utility": "Intentionally vague prompt like 'Make two qubits the same somehow.' Tests whether LLM asks for clarification or picks a valid interpretation.",
      "Difficulty": "Intermediate",
      "prompt": "As as Cirq circuit, Make two qubits the same somehow.",
      "prompt_style": "conversational",
      "num_qubits": "2+",
      "expected_gates": ["TBD — model chooses"],
      "expected_behavior": "Model uses its own reasoning skills to create entangled qubits."
      "resulted_code": "```python\nimport cirq\n\n# Create two qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n# Apply Hadamard gate to q0\ncircuit.append([cirq.H(q0)])\n\n# Apply controlled-NOT gate to q1 with q0 as the control qubit\ncircuit.append([cirq.CNOT(q0, q1)])\n\n# Print the circuit\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\n\n# Print the final state of the qubits\nprint(result.state)\n```", 
      "validated_verdict": "invalid code, added simulation, however correct circuit, generated bell state", 
    },
    {
      "Circuit": "Ambiguous Entanglement Prompt",
      "Description of Utility": "Intentionally vague prompt like 'Make two qubits the same somehow.' Tests whether LLM asks for clarification or picks a valid interpretation.",
      "Difficulty": "Intermediate",
      "prompt": "Create a cirq circuit where measuring one qubit tells you something about the other.",
      "prompt_style": "narrative",
      "num_qubits": "2+",
      "expected_gates": ["TBD — model chooses"],
      "expected_behavior": "Model uses its own reasoning skills to create entangled qubits."
      "resulted_code": "```python\nimport cirq\nimport numpy as np\n\n# Define two qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\n\n# Create a circuit with a Hadamard gate on each qubit to prepare an entangled state\ncircuit = cirq.Circuit(\n    cirq.H(q0),\n    cirq.H(q1),\n    cirq.measure(q0, key='q0'),\n    cirq.measure(q1, key='q1')\n)\n\n# Create a Simulator and run the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\n\n# Print the results\nprint(result.histogram(key='q0'))\nprint(result.histogram(key='q1'))\n",
      "validated_verdict:": "invalid code, added simulation, correct circuit, made uniform superposition state.",
    },
    {
      "Circuit": "Quantum Teleportation",
      "Description of Utility": "Transfers an unknown qubit state using entanglement and classical communication. Three-qubit protocol beyond project scope — stretch goal.",
      "Difficulty": "Expert",
      "Expected Gates": "H, CNOT, CNOT, H, M, X, Z",
      "Status": "Haven't Touched"
    },
    {
      "Circuit": "Toffoli Gate (CCNOT)",
      "Description of Utility": "Three-qubit gate — flips target only if both control qubits are |1⟩. Universal for classical reversible computation.",
      "Difficulty": "Advanced",
      "Expected Gates": "CCNOT",
      "Status": "Haven't Touched"
    },
    {
      "Circuit": "Pauli-Z Gate",
      "Description of Utility": "Applies a phase flip — leaves |0⟩ unchanged and flips the sign of |1⟩. Used in phase kickback and error correction.",
      "Difficulty": "Beginner",
      "prompt": "Apply a Pauli-Z gate to a single qubit initialized in the |+⟩ computational basis state and return the resulting Cirq circuit.",
      "prompt_style": "technical",
      "num_qubits": 1,
      "expected_gates": ["Z"],
      "expected_behavior": "After Z is applied the qubits phase flips."
      "resulted code":""```python\nimport cirq\n\n# Initialize a qubit\nqubit = cirq.LineQubit(0)\n\n# Define the initial state |+⟩\ninitial_state = (cirq.X(qubit) @ cirq.X(qubit)) ** 0.5\n\n# Create a circuit with the initial state\ncircuit = cirq.Circuit(initial_state)\n\n# Apply a Pauli-Z gate to the qubit\npauli_z_gate = cirq.Z(qubit)\ncircuit.append(cirq.measure(qubit, key="z"),\n               cirq.measure(qubit, key=\'full\'))\n\ncircuit.append(pauli_z_gate)\n\nprint(circuit)\n```",
      "validated_verdict:": "invalid circuit, tried to innitialize the innitial state wrong, write idea ig.",
    },
    {
      "Circuit": "Pauli-Z Gate",
      "Description of Utility": "Applies a phase flip — leaves |0⟩ unchanged and flips the sign of |1⟩. Used in phase kickback and error correction.",
      "Difficulty": "Beginner",
      "prompt": "Create a cirq circuit that applied a phase flip to a qubit. Implement the Z gate.",
      "prompt_style": "conversational",
      "num_qubits": 1,
      "expected_gates": ["Z"],
      "expected_behavior": "After Z is applied the qubits phase flips."
      "resulted_code": "```python\nimport cirq\n\n# Create a qubit\nqubit = cirq.LineQubit(0)\n\n# Create the Z gate\nz_gate = cirq.Z(qubit)\n\n# Print the Z gate\nprint("Z gate: ", z_gate)\n\n# Apply the Z gate to the qubit\ncircuit = cirq.Circuit(z_gate)\n\n# Print the circuit\nprint("Circuit: ", circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\n\n# Print the expected output\nprint("Expected output (for the state |0>): ", 0 * cirq.bit_string_to_list(\'1\'))\nprint("Simulator output: ", result.output())\n```", 
      "validated_verdict":  "Correct Circuit, hallucinated simulation resulted to invalid code", 
    },
    {
      "Circuit": "Pauli-Z Gate",
      "Description of Utility": "Applies a phase flip — leaves |0⟩ unchanged and flips the sign of |1⟩. Used in phase kickback and error correction.",
      "Difficulty": "Beginner",
      "prompt": "Create a cirq circuit that models a phase flip, mapping |+⟩ → |−⟩ = (|0⟩ - |1⟩)/√2.",
      "prompt_style": "narrative",
      "num_qubits": 1,
      "expected_gates": ["Z"],
      "expected_behavior": "After Z is applied the qubits phase flips."
      "resulted_code": "```python\nimport cirq\n\n# Define the qubits\nq = cirq.LineQubit(0)\n\n# Define the phase flip operation\nphase_flip = cirq.X(q)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(phase_flip, repetitions=1)\n\n# Print the result\nprint(result)\n```",
      "validated_verdict:": "Invalid Code, Incorrect Gates", 
    },
    {
      "Circuit": "Bell State with Measurement",
      "Description of Utility": "Bell state with measurement operations added to both qubits. Tests whether LLM correctly appends measurements when prompted.",
      "Difficulty": "Beginner",
      "prompt": "Prepare the Bell state |Phi+> on two qubits as a Cirq circuit and add mesurement gates to both qubits.",
      "prompt_style": "technical",
      "num_qubits": 2,
      "expected_gates": ["H", "CNOT", "M"],
      "expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability."
      "resulted code": "import cirq\n\n# Create a simulator\nsimulator = cirq.Simulator()\n\n# Define a circuit for a Bell state |Phi+>\nq0, q1 = cirq.LineQubit.range(2)\n\n# Apply Hadamard gate to qubit 0\ncircuit = cirq.Circuit(\n    cirq.H(q0),\n    cirq.CNOT(q0, q1)\n)\n\n# Add measurement gates\ncircuit.append(cirq.measure(q0, key="q0"))\ncircuit.append(cirq.measure(q1, key="q1"))\n\n# Run the circuit\nresult = simulator.run(circuit, repetitions=100)\n\n# Print the measurement outcomes\nprint(result.histogram(key="q0"))\nprint(result.histogram(key="q1"))\n",
      "validated verdict": "Correct circuit, simulator, didn't print circuit"
    },
    {
      "Circuit": "Bell State with Measurement",
      "Description of Utility": "Bell state with measurement operations added to both qubits. Tests whether LLM correctly appends measurements when prompted.",
      "Difficulty": "Beginner",
      "prompt": "Generate a quantum circuit that produces 00 and 11 with equal probibility and measure both qubits",
      "prompt_style": "narrative",
      "num_qubits": 2,
      "expected_gates": ["H", "CNOT", "M"],
      "expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability.",
      "resulted_code": "import cirq\nimport numpy as np\n\n# Define two qubits\nq0 = cirq.GridQubit(0, 0)\nq1 = cirq.GridQubit(0, 1)\n\n# Create a circuit with Hadamard gate on the first qubit and CNOT gate to produce 00 and 11 with equal probablility,\n# then append identity with pair of qubits to preserve superposition length in both the states 00 and 11\ncircuit = cirq.Circuit(\n    cirq.H(q0),\n    cirq.CNOT(q0, q1),\n    cirq.CX(q0,q1)\n)\n\n# Perform the circuit and measure both qubits\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=10000)\n\n# Print the results\nfor outcome in cirq.measure([q0, q1], key=\'outcome\'):\n    print(f"qubits are: {outcome}")\nprint("The probability of getting 00 is: ", result.histogram(key=\'outcome\')[0]/10000)\nprint("The probability of getting 11 is: ", result.histogram(key=\'outcome\')[1]/10000)\n", 
    "validated_verdict": "invalid code, spacing error"
    },
    {
      "Circuit": "Bell State with Measurement",
      "Description of Utility": "Bell state with measurement operations added to both qubits. Tests whether LLM correctly appends measurements when prompted.",
      "Difficulty": "Beginner",
      "prompt": "Entangle 2 qubits as a Cirq Circuit and Measure each qubit.",
      "prompt_style": "conversational",
      "num_qubits": 2,
      "expected_gates": ["H", "CNOT", "M"],
      "expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability."
      "resulted_code":  "python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n# Add a Hadamard gate to qubit0 to create superposition\ncircuit.append(cirq.H(qubit0))\n\n# Add a CNOT gate to entangle qubit0 and qubit1\ncircuit.append(cirq.CNOT(qubit0, qubit1))\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nprint("\\nFinal state:)\nprint(result.final_state)\n",
      "validated_verdict:":"invalid code, correct cirq but measurement was wrong, tried to simulte w/o proper measurements",
    },
    {
      "Circuit": "RY Rotation",
      "Description of Utility": "Rotates a qubit around the Y-axis by a specified angle. Common in ansatz layers for quantum ML.",
      "Difficulty": "intermediate",
      "prompt": "Apply an RY rotation gate with angle θ = π/2 to a single qubit initialized in the |+⟩ state (after a Hadamard) and return the resulting Cirq circuit.",
      "prompt_style": "technical",
      "num_qubits": 1,
      "expected_gates": ["RY(theta)"],
      "expected_behavior": "After RY is applied the qubits phase flips."
      "resulted code":"```python\nimport cirq\n\n# Initialize the qubit\nqubit = cirq.LineQubit(0)\n\n# Define the rotation angle\ntheta = cirq.PI / 2\n\n# Apply the Hadamard gate followed by the RY rotation gate\ncircuit = cirq.Circuit(cirq.H(qubit), cirq.Ry(theta)(qubit))\n\n# Print the circuit\nprint(circuit)\n",
      "validated_verdict": "correct circuit, hallucinated pi in cirq instead of numpy", 
    },
    {
      "Circuit": "RY Rotation",
      "Description of Utility": "Rotates a qubit around the Y-axis by a specified angle. Common in ansatz layers for quantum ML.",
      "Difficulty": "intermediate",
      "prompt": "Create a cirq circuit that rotates a qubit around the Y axis with angle θ = π/2. Implement a H gate, and then the RY gate.",
      "prompt_style": "conversational",
      "num_qubits": 1,
      "expected_gates": ["RY(theta)"],
      "expected_behavior": "After RY is applied the qubits phase flips."
      "resulted_code": "python\nimport cirq\n\n# Define the RY gate from the qelib1 library\nqubits = cirq.LineQubit.range(1)\nRY_gate = cirq.Ry(cirq.named_euler_angles("-pi/2", "0", "0"))\n\n# Create the circuit\ncircuit = cirq.Circuit(\n    cirq.H(qubits[0]),  # Hadamard gate\n    RY_gate(qubits[0])   # Rotate around Y axis by π/2\n)\n\nprint(circuit)\n", 
      "validated_verdict":  "correct circuit, hallucinated 'named_euler_angles' to define rotation, right idea", 
    },
    {
      "Circuit": "RY Rotation",
      "Description of Utility": "Rotates a qubit around the Y-axis by a specified angle. Common in ansatz layers for quantum ML.",
      "Difficulty": "intermediate",
      "prompt": "Create a cirq circuit that models a phase flip, mapping |0⟩ + |1⟩ → cos(pi/2)|0⟩ + sin(pi/2)|1⟩ ",
      "prompt_style": "narrative",
      "num_qubits": 1,
      "expected_gates": ["RY(theta)"],
      "expected_behavior": "After RY is applied the qubits phase flips."
      "resulted_code": "```python\nimport cirq\nimport cirq谷山\n\n# Define the circuit\nqubit = cirq.GridQubit(0, 0)\ncircuit = cirq.Circuit(\n    cirq.H(qubit),\n    cirq.measure(qubit, key=\'z\')\n)\n\n# Define the initial state\nstate = [0, 1]  # 0: |0⟩, 1: |1⟩\n\n\n# Apply the circuit to the state\nresult = cirq.SimulateAllOutputs(affine_triangular()).simulate(circuit, qubit).histogram(key=\'z\')\nprint(result.histogram(key="z"))\n\n# Output: \n# Histogram({\'0\': 8388607, \'1\': 8388607, \'\': 0})\n\n# This is close to the expected output, which would be:\n# Histogram({\'0\': 0, \'1\': 8388607, \'\': 0})\n# but due to numerical inaccuracies in the simulator, the probability of 0 is very small\n```",
      "validated_verdict:": "invalid code, halluciated code, slop, not close to right idea", 
    },
    {
      "Circuit": "GHZ State (3-qubit)",
      "Description of Utility": "Entangles three qubits into a superposition of |000⟩ and |111⟩. Extends Bell state logic to three qubits.",
      "Difficulty": "Intermediate",
      "prompt": "Prepare a three-qubit GHZ state by applying a Hadamard gate to qubit 0 followed by CNOT gates with qubit 0 as control targeting qubit 1, and qubit 0 as control targeting qubit 2. Return the resulting Cirq circuit.",
      "prompt_style": "technical",
      "num_qubits": 3,
      "expected_gates": ["H, CNOT, CNOT"],
      "expected_behavior": "Measurement outcomes 000 and 111 should dominate with roughly equal probability",
      "resulted_code": "python\nimport cirq\n\n# Define a function to create a GHZ circuit with three qubits\ndef create_ghz_circuit(num_qubits=3):\n    # Create a Circuit object\n    circuit = cirq.Circuit()\n\n    # Create qubits\n    qubits = [cirq.LineQubit(i) for i in range(num_qubits)]\n\n    # Add Hadamard gate to qubit 0\n    circuit.append(cirq.H(qubits[0]))\n\n    # Add CNOT gates controlled by qubit 0 targeting qubits 1 and 2\n    circuit.append(cirq.CNOT(qubits[0], qubits[1]))\n    circuit.append(cirq.CNOT(qubits[0], qubits[2]))\n\n    return circuit\n\n# Create a GHZ circuit with three qubits\nghz_circuit = create_ghz_circuit()\n\n# Print the circuit\nprint("GHZ Circuit:")\nprint(ghz_circuit)\n", 
      "validated verdict": "success"
    },
    {
      "Circuit": "GHZ State (3-qubit)",
      "Description of Utility": "Entangles three qubits into a superposition of |000⟩ and |111⟩. Extends Bell state logic to three qubits.",
      "Difficulty": "Intermediate",
      "prompt": "Generate a cirq quantum circuit that produces 000 and 111 with equal probibility.",
      "prompt_style": "narrative",
      "num_qubits": 3,
      "expected_gates": ["H, CNOT, CNOT"],
      "expected_behavior": "Measurement outcomes 000 and 111 should dominate with roughly equal probability.",
      "resulted_code": "```python\nimport cirq\n\n# Create two qubits\nq0, q1, q2 = cirq.GridQubit.rect(1, 3, 0, 0, 1, 2)\n\n# Define the circuit\ncircuit = cirq.Circuit(\n    # Apply Hadamard gates to create a superposition state\n    cirq.H(q0),\n    cirq.H(q1),\n    cirq.H(q2),\n\n    # Controlled-NOT gates to entangle the qubits\n    cirq.CX(q1, q0),\n    cirq.CX(q2, q1),\n    cirq.CX(q1, q0),\n\n    # Measure the qubits\n    cirq.measure(q0, q1, q2)\n)\n\n# Simulate the circuit\nresult = cirq.Simulator().run(circuit, repetitions=1000)\n\n# Print the results\nprint(cirq统计=result.histogram(key=cirq_ALL_MEASURED))\n",
    "validated_verdict": "invalid code, used too many aguments to innitialize qubits, right idea, print slop",
    },
    {
      "Circuit": "GHZ State (3-qubit)",
      "Description of Utility": "Entangles three qubits into a superposition of |000⟩ and |111⟩. Extends Bell state logic to three qubits.",
      "Difficulty": "Intermediate",
      "prompt": "As a Cirq Circuit, Entangle 3 qubits.",
      "prompt_style": "conversational",
      "num_qubits": 3,
      "expected_gates": ["H, CNOT, CNOT"],
      "expected_behavior": "Measurement outcomes 000 and 111 should dominate with roughly equal probability."
      "resulted_code": "python\nimport cirq\n\n# Define the qubits\nq1, q2, q3 = cirq.LineQubit.range(3)\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n# GHZ (Greenberger-Horne-Zeilinger) state is an example of 3-qubit entanglement\n# The GHZ state is given by the operation:  |000> -> |000> + i|111>\ncircuit.append([\n    cirq.ISWAP(q1, q2)**0.5,\n    cirq.ISWAP(q1, q3)**0.5,\n])\n\n# Print the circuit\nprint(circuit)\n\n# Simulate the circuit and measure the outcomes\nsim = cirq.Simulator()\nresult = sim.simulate(circuit, repetitions=1000)\n\n# Print the statistics of the outcomes\nprint(cirq.density_matrix(result).full()).round(3)\n"
      "validated_verdict:": "invalid_code, hallucinated agrument for simulation, wrong gates, wrong idea",
    },
    {
      "Circuit": "GHZ State with Measurement",
      "Description of Utility": "GHZ circuit with measurements on all three qubits. Semantic check: outcomes should be only 000 and 111.",
      "Difficulty": "Intermediate",
      "prompt": "Prepare a three-qubit GHZ state by applying a Hadamard gate to qubit 0 followed by CNOT gates with qubit 0 as control targeting qubit 1, and qubit 0 as control targeting qubit 2. After, apply a measurement gate on each qubit. Return the resulting Cirq circuit.",
      "prompt_style": "technical",
      "num_qubits": 3,
      "expected_gates": ["H, CNOT, CNOT, M"],
      "expected_behavior": "Measurement outcomes 000 and 111 should dominate with roughly equal probability",
      "resulted_code": "python\nimport cirq\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n# Define the qubits\nq0, q1, q2 = cirq.LineQubit.range(3)\n\n# Prepare the GHZ state by applying a Hadamard gate to qubit 0\ncircuit.append(cirq.H(q0))\n\n# Apply CNOT gates with qubit 0 as control\ncircuit.append(cirq.CNOT(q0, q1))\ncircuit.append(cirq.CNOT(q0, q2))\n\n# Apply a measurement gate on each qubit\ncircuit.append(cirq.measure(q0, key='q0'))\ncircuit.append(cirq.measure(q1, key='q1'))\ncircuit.append(cirq.measure(q2, key='q2'))\n\n# Print the circuit\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\n\n# Print the measurement outcomes\nprint(result)\n", 
      "validated verdict": "right idea,right gates, added simulation"
    },
    {
      "Circuit": "GHZ State with Measurement",
      "Description of Utility": "GHZ circuit with measurements on all three qubits. Semantic check: outcomes should be only 000 and 111.",
      "Difficulty": "Intermediate",
      "prompt": "Generate a cirq quantum circuit that produces 000 and 111 with equal probibility and measures each qubit.",
      "prompt_style": "narrative",
      "num_qubits": 3,
      "expected_gates": ["H, CNOT, CNOT, M"],
      "expected_behavior": "Measurement outcomes 000 and 111 should dominate with roughly equal probability.",
      "resulted_code": "python\nimport numpy as np\nimport cirq\n\n# Create three qubits\nqubits = [cirq.GridQubit(0, i) for i in range(3)]\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n# Add a Hadamard gate to each qubit to create a superposition\nfor qubit in qubits:\n    circuit.append(cirq.H(qubit))\n\n# Add a controlled-NOT gate between each pair of qubits to entangle them\ncircuit.append([\n    cirq.CNOT(qubits[0], qubits[1]),\n    cirq.CNOT(qubits[1], qubits[2])\n])\n\n# Add a CircuitDuplicator to duplicate the circuit and send all qubits through two different Hadamard gates\ncircuit.append([\n    cirq.CircuitDuplicator()((cirq.H(qubit), cirq.{Hmant(qubit)}))\n    for qubit in qubits\n])\n\n# Add measurements to the circuit\ncircuit.append([\n    cirq.measure(qubit, key=f'q{qubit}')\n    for qubit in qubits\n])\n\n# Simulate the circuit 1000 times\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\n\n# Print the results\nprint(result)\n",
    "validated_verdict": "invalid code, right idea, hallucinated code",
    },
    {
      "Circuit": "GHZ State with Measurement",
      "Description of Utility": "GHZ circuit with measurements on all three qubits. Semantic check: outcomes should be only 000 and 111.",
      "Difficulty": "Intermediate",
      "prompt": "As a Cirq Circuit, Entangle 3 qubits and measure each qubit.",
      "prompt_style": "conversational",
      "num_qubits": 3,
      "expected_gates": ["H, CNOT, CNOT, M"],
      "expected_behavior": "Measurement outcomes 000 and 111 should dominate with roughly equal probability."
      "resulted_code": "python\nimport cirq\n\n# Create a Simulator engine to run the circuit\nsimulator = cirq.Simulator()\n\n# Create 3 qubits (quby-1, quby-2, quby-3)\nquby_1 = cirq.LineQubit(0)\nquby_2 = cirq.LineQubit(1)\nquby_3 = cirq.LineQubit(2)\n\n# Create the circuit\ncircuit = cirq.Circuit(\n    # Apply the Hadamard gate to the first qubit\n    cirq.H(quby_1),\n    \n    # Apply a Controlled-NOT gate between quby_1 and quby_2\n    cirq.CNOT(quby_1, quby_2),\n    \n    # Apply a Controlled-NOT gate between quby_2 and quby_3\n    cirq.CNOT(quby_2, quby_3),\n\n    # Measure the qubits\n    cirq.measure(quby_1, quby_2, quby_3, key='result'),\n)\n\n# Run the circuit\nresult = simulator.run(circuit, repetitions=2)\nprint(result)\n"
      "validated_verdict:": "right circuit, right gates, right idea, ran simulation instead of circuit",
    },
    {
      "Circuit": "SWAP Gate",
      "Description of Utility": "Exchanges the states of two qubits. Used in routing algorithms and circuit compilation.",
      "Difficulty": "Intermediate",
      "prompt": "Construct a two-qubit Cirq circuit implementing a SWAP gate between qubit 0 initialized in |1⟩ and qubit 1 initialized in |0⟩. Return the resulting Cirq circuit.",
      "prompt_style": "technical",
      "num_qubits": 1,
      "expected_gates": ["SWAP"],
      "expected_behavior": "The SWAP gate exchanges the full quantum states of the two qubits. With qubit 0 starting in |1⟩ and qubit 1 in |0⟩, the final state after SWAP is |01⟩ — qubit 0 is now |0⟩ and qubit 1 is now |1⟩. The state vector is [0, 1, 0, 0]."
      "resulted code":"python\nimport cirq\n\nq0, q1 = cirq.LineQubit.range(2)\n\ncircuit = cirq.Circuit(\n    cirq.X(q0),\n    cirq.X(q1),\n    cirq.CX(q0, q1),\n    cirq.CX(q0, q1),\n    cirq.X(q1)\n)\n\nprint(circuit)\n",
      "validated verdict: "wrong gates, tried to decompose swap, failed"
    },
    {
      "Circuit": "SWAP Gate",
      "Description of Utility": "Exchanges the states of two qubits. Used in routing algorithms and circuit compilation.",
      "Difficulty": "Intermediate",
      "prompt": "Create a cirq circuit that swaps the states of two qubits.",
      "prompt_style": "conversational",
      "num_qubits": 1,
      "expected_gates": ["SWAP"],
      "expected_behavior": "The SWAP gate exchanges the full quantum states of the two qubits. With qubit 0 starting in |1⟩ and qubit 1 in |0⟩, the final state after SWAP is |01⟩ — qubit 0 is now |0⟩ and qubit 1 is now |1⟩. The state vector is [0, 1, 0, 0]."
      "resulted_code": "```c\nimport cirq\n\n# Define the qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\n\n# Create a cirq circuit\ncircuit = cirq.Circuit()\n\n# Define a function for a CNOT gate, which swaps the state of two qubits when controlled by another qubit\ndef swap_qubits(q0, q1):\n    # First, apply a CNOT gate to swap the states of q0 and q1. To do this, we "control" q1 and apply a NOT gate to q0.\n    circuit.append(cirq.CCX(q0, q1))\n    # Then, we control q0 and apply a NOT gate to q1. This effectively flips the swap applied in the previous step.\n    circuit.append(cirq.CCX(q1, q0))\n\n# Apply the swap function to swap the state of q0 and q1\nswap_qubits(q0, q1)\n\n# Print the circuit\nprint(circuit)\n\n# Apply the circuit on a simulator\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit, cirq.LindbladSimulator(np.random.default_rng()))\ninfo = round(result.final_statenapshot[0][0].real, 3)\n\nassert info < 1, f\' Error in final state that was calculated, expected less than 1 but was {result.final_statenapshot[0][0].real}\'  \nassert info > 0, f\' Error in final state that was calculated, expected greater than 0 but was {result.final_statenapshot[0][0].real}\'   \nassert info > 0.5, f\' Error in final state that was calculated, expected greater than 0.5 but was {result.final_statenapshot[0][0].real}\'   \nassert info < 1, f\' Error in final state that was calculated, expected less than 1 but was {result.final_statenapshot[0][0].real}\'   \n\nprint(f"The final state of the outcome |00| is {info}.")\n```", 
      "validated_verdict": "invalid, wrong circuit, wrong gates, hallucinated code, slop", 
    },
    {
      "Circuit": "SWAP Gate",
      "Description of Utility": "Exchanges the states of two qubits. Used in routing algorithms and circuit compilation.",
      "Difficulty": "Intermediate",
      "prompt": "Create a cirq circuit that models a swap between the states of two qubits.",
      "prompt_style": "narrative",
      "num_qubits": 1,
      "expected_gates": ["SWAP"],
      "expected_behavior": "The SWAP gate exchanges the full quantum states of the two qubits. With qubit 0 starting in |1⟩ and qubit 1 in |0⟩, the final state after SWAP is |01⟩ — qubit 0 is now |0⟩ and qubit 1 is now |1⟩. The state vector is [0, 1, 0, 0]."
      "resulted_code": "python\nimport cirq\n\n# Create two qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\n\n# Define the circuit using the SWAP gate\ncircuit = cirq.Circuit(cirq.swap(q0, q1))\ncircuit.append(cirq.measure(q0, q1, key='q0'+','+'q1'))\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\nprint(result.histogram(key='q0'+','+'q1'))\n",
      "validated_verdict:": "invalid circuit, hallucinated swap gate, right idea",
    },
    {
      "Circuit": "Single-Qubit Measurement",
      "Description of Utility": "Puts qubit in superposition then measures. Should yield ~50/50 outcomes. Tests measurement-only prompts.",
      "Difficulty": "Beginner",
      "prompt": "Apply a Hadamard gate to a single qubit initialized in the |0⟩ computational basis state and return the resulting Cirq circuit. After, apply a measurement gate.",
      "prompt_style": "technical",
      "num_qubits": 1,
      "expected_gates": ["H, M"],
      "expected_behavior": "After H is applied the qubit is put into superposition"
      "resulted code:```import cirq\n\n# Define the qubit\nqubit = cirq.LineQubit(0)\n\n# Initialize the qubit in the |0⟩ state\ncircuit = cirq.Circuit(cirq.X(qubit))\n\n# Apply a Hadamard gate to the qubit\ncircuit += cirq.H(qubit)\n\n# Apply a measurement gate\ncircuit += cirq.measure(qubit, key='result')\n\n# Run the circuit on the simulator\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\n\nprint(result.histogram(key='result'))\n```",
      "validated verdict": "mostly correct circuit, added an x gate to prepare circuit when it wasn't necessary."
    },
    {
      "Circuit": "Single-Qubit Measurement",
      "Description of Utility": "Puts qubit in superposition then measures. Should yield ~50/50 outcomes. Tests measurement-only prompts.",
      "Difficulty": "Beginner",
      "prompt": "In a cirq circuit put a qubit in superposition and then meausure it.",
      "prompt_style": "conversational",
      "num_qubits": 1,
      "expected_gates": ["H, M"],
      "expected_behavior": "After H is applied the qubit is put into superposition"
      "resulted_code": "```python\nimport cirq\n\n# Create a qubit\nqubit = cirq.LineQubit(0)\n\n# Put the qubit in superposition\nsuperposition = cirq.X(qubit)\n\n# Create a circuit to put the qubit in superposition and then measure it\ncircuit = cirq.Circuit(superposition, cirq.measure(qubit))\n\n# Print the circuit\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\n\n# Print the measurement result\nprint(result)\n```", 
      "validated_verdict": "wrong code, used x gate instead of H to put the qubit in super position", 
    },
    {
      "Circuit": "Single-Qubit Measurement",
      "Description of Utility": "Puts qubit in superposition then measures. Should yield ~50/50 outcomes. Tests measurement-only prompts.",
      "Difficulty": "Beginner",
      "prompt": "Generate a cirq quantum circuit that makes a qubit's state 50/50 |1> and |0> and add a measurement gate afterward",
      "prompt_style": "narrative",
      "num_qubits": 1,
      "expected_gates": ["H, M"],
      "expected_behavior": "After H is applied the qubit is put into superposition"
      "resulted_code": "```python\nimport cirq\n\n# Specify the simulator to use\nsimulator = cirq.Simulator()\n\n# Define the qubit to use\nqubit = cirq.LineQubit(0)\n\n# Create a circuit and add a Hadamard gate to make the qubit's state 50/50 |1> and |0>\ncircuit = cirq.Circuit(\n    cirq.H(qubit)  # Hadamard gate to make the qubit's state 50/50 |1> and |0>\n)\n\n# Add a measurement gate to the circuit\ncircuit.append(cirq.measure(qubit, key='result'))\n\n# Run the circuit\nresult = simulator.run(circuit, repetitions=1000)\n\n# Print the measurement results\nprint(result.histogram(key='result', bins=2))\n```",
      "validated_verdict:": "correct circuit, simulates instead of prints, halucinated argument within simulation",
    },
    {
      "Circuit": "RZ Rotation",
      "Description of Utility": "Rotates a qubit around the Z-axis by a specified angle. Used in variational and parameterized circuits.",
      "Difficulty": "intermediate",
      "prompt": "Apply an RZ rotation gate with angle θ = π/2 to a single qubit initialized in the |+⟩ state (after a Hadamard) and return the resulting Cirq circuit.",
      "prompt_style": "technical",
      "num_qubits": 1,
      "expected_gates": ["RZ(theta)"],
      "expected_behavior": "After RZ is applied the qubits phase flips."
      "resulted code":"```python\nimport cirq\n\n# Define the qubit\nq = cirq.LineQubit(0)\n\n# Initialize the circuit with the Hadamard gate\ncircuit = cirq.Circuit(cirq.H(q))\n\n# Apply an RZ rotation gate with angle θ = π/2\ncircuit.append(cirq.Rz(cirq.pi/2).on(q))\n\n# Print the circuit\nprint(circuit)\n",
      "validated_verdict": "correct circuit, hallucinated pi in cirq instead of numpy", 
    },
    {
      "Circuit": "RZ Rotation",
      "Description of Utility": "Rotates a qubit around the Z-axis by a specified angle. Used in variational and parameterized circuits.",
      "Difficulty": "intermediate",
      "prompt": "Create a cirq circuit that rotates a qubit around the Z axis with angle θ = π/2. Implement the RZ gate.",
      "prompt_style": "conversational",
      "num_qubits": 1,
      "expected_gates": ["RX(theta)"],
      "expected_behavior": "After X is applied the qubits phase flips."
      "resulted_code": "python\nimport cirq\n\n# Define a qubit\nqubit = cirq.LineQubit(0)\n\n# Define the rotation angle\ntheta = cirq.pi / 2\n\n# Create a circuit that rotates the qubit around the Z axis by theta\ncircuit = cirq.Circuit(cirq.Rz(theta).on(qubit))\n\n# Print the circuit\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nprint("Final State:")\nprint(result.state)\n", 
      "validated_verdict":  "correct circuit, hallucinated pi in cirq instead of numpy", 
    },
    {
      "Circuit": "RZ Rotation",
      "Description of Utility": "Rotates a qubit around the Z-axis by a specified angle. Used in variational and parameterized circuits.",
      "Difficulty": "intermediate",
      "prompt": "Create a cirq circuit that models a phase flip, mapping |1⟩ → i|1⟩",
      "prompt_style": "narrative",
      "num_qubits": 1,
      "expected_gates": ["RZ(theta)"],
      "expected_behavior": "After Z is applied the qubits phase flips."
      "resulted_code": "```python\nimport cirq\n\n# Define a qubit\nqubit = cirq.LineQubit(0)\n\n# Define the rotation angle\ntheta = cirq.pi / 2\n\n# Create a circuit that rotates the qubit around the Z axis by theta\ncircuit = cirq.Circuit(cirq.Rz(theta).on(qubit))\n\n# Print the circuit\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nprint("Final State:")\nprint(result.state)\n```",
      "validated_verdict:": "correct circuit, hallucinated pi in cirq instead of numpy", 
    },
    {
      "Circuit": "Quantum Fourier Transform (3-qubit)",
      "Description of Utility": "Computes the discrete Fourier transform on quantum amplitudes. Subroutine in Shor's algorithm and phase estimation. More complex than project scope.",
      "Difficulty": "Expert",
      "Expected Gates": "H, CPHASE, SWAP",
      "Status": "Haven't Touched"
    }
  ]
}

SyntaxError: invalid character '⁺' (U+207A) (518843184.py, line 36)

In [40]:
llm_tv = """import cirq\n\n# Create two qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\n\n# Apply the CNOT gate to q0 and q1\ncircuit = cirq.Circuit(\n    cirq.X(q0) ** 0.5,  # Optional, applies a Hadamard gate to q0\n    cirq.CNOT(q0, q1),\n    cirq.MCTLayers()[0][cirq.MeasurementGate(1)()(q0)],  # Measure q0\n    cirq.measure(q0, key='q0'),\n    cirq.measure(q1, key='q1')\n)\n\n# Simulate the circuit and print the output\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\n\nprint(result.histogram(key='q0'))\n"""

validate_circuit(llm_tv, variational_circuit)

{'syntax': {'passed': False,
  'error': "module 'cirq' has no attribute 'MCTLayers'"},
 'verdict': 'invalid_code'}

In [31]:
q0,q1 = cirq.LineQubit.range(2)

s_circuit = cirq.Circuit()
s_circuit += cirq.SWAP(q0,q1)
print(s_circuit)

llm_s = """import cirq\n\n# Create two qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\n\n# Define the circuit using the SWAP gate\ncircuit = cirq.Circuit(cirq.swap(q0, q1))\ncircuit.append(cirq.measure(q0, q1, key='q0'+','+'q1'))\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\nprint(result.histogram(key='q0'+','+'q1'))\n"""

validate_circuit(llm_s, s_circuit)

0: ───×───
      │
1: ───×───


{'syntax': {'passed': False, 'error': "module 'cirq' has no attribute 'swap'"},
 'verdict': 'invalid_code'}

In [25]:
q0,q1,q2 = cirq.LineQubit.range(3)

ghzm_circuit = cirq.Circuit()
ghzm_circuit += cirq.H(q0)
ghzm_circuit += cirq.CNOT(q0, q1)
ghzm_circuit += cirq.CNOT(q1, q2)
q0, q1, q2 = cirq.LineQubit.range(3)

ghzm_circuit = cirq.Circuit()
ghzm_circuit += cirq.H(q0)
ghzm_circuit += cirq.CNOT(q0, q1)
ghzm_circuit += cirq.CNOT(q1, q2)

ghzm_circuit.append(cirq.measure(q0, key='q0'))
ghzm_circuit.append(cirq.measure(q1, key='q1'))
ghzm_circuit.append(cirq.measure(q2, key='q2'))

print(ghzm_circuit)

llm_ghzm = """import cirq\n\n# Create a Simulator engine to run the circuit\nsimulator = cirq.Simulator()\n\n# Create 3 qubits (quby-1, quby-2, quby-3)\nquby_1 = cirq.LineQubit(0)\nquby_2 = cirq.LineQubit(1)\nquby_3 = cirq.LineQubit(2)\n\n# Create the circuit\ncircuit = cirq.Circuit(\n    # Apply the Hadamard gate to the first qubit\n    cirq.H(quby_1),\n    \n    # Apply a Controlled-NOT gate between quby_1 and quby_2\n    cirq.CNOT(quby_1, quby_2),\n    \n    # Apply a Controlled-NOT gate between quby_2 and quby_3\n    cirq.CNOT(quby_2, quby_3),\n\n    # Measure the qubits\n    cirq.measure(quby_1, quby_2, quby_3, key='result'),\n)\n\nprint(circuit)"""

validate_circuit(llm_ghzm, ghzm_circuit)

0: ───H───@───M('q0')─────────────
          │
1: ───────X───@─────────M('q1')───
              │
2: ───────────X─────────M('q2')───
0: ───H───@───────M('result')───
          │       │
1: ───────X───@───M─────────────
              │   │
2: ───────────X───M─────────────
0: ───H───@───────M('result')───
          │       │
1: ───────X───@───M─────────────
              │   │
2: ───────────X───M─────────────


KeyError: 'result'

In [15]:
q0,q1,q2 = cirq.LineQubit.range(3)

ghz_circuit = cirq.Circuit()
ghz_circuit += cirq.H(q0)
ghz_circuit += cirq.CNOT(q0, q1)
ghz_circuit += cirq.CNOT(q1, q2)
print(ghz_circuit)

llm_ghz = """import cirq\n\n# Define the qubits\nq1, q2, q3 = cirq.LineQubit.range(3)\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n# GHZ (Greenberger-Horne-Zeilinger) state is an example of 3-qubit entanglement\n# The GHZ state is given by the operation:  |000> -> |000> + i|111>\ncircuit.append([\n    cirq.ISWAP(q1, q2)**0.5,\n    cirq.ISWAP(q1, q3)**0.5,\n])\n\n# Print the circuit\nprint(circuit)\n\n# Simulate the circuit and measure the outcomes\nsim = cirq.Simulator()\nresult = sim.simulate(circuit, repetitions=1000)\n\n# Print the statistics of the outcomes\nprint(cirq.density_matrix(result).full()).round(3)\n"""

validate_circuit(llm_ghz, ghz_circuit)


0: ───H───@───────
          │
1: ───────X───@───
              │
2: ───────────X───
0: ───iSwap───────iSwap───────
      │           │
1: ───iSwap^0.5───┼───────────
                  │
2: ───────────────iSwap^0.5───


{'syntax': {'passed': False,
  'error': "SimulatesFinalState.simulate() got an unexpected keyword argument 'repetitions'"},
 'verdict': 'invalid_code'}

In [ ]:
import numpy as np
q = cirq.LineQubit(0)

ry_circuit = cirq.Circuit(
    cirq.H(q),
    cirq.ry(np.pi / 2)(q)
)

print(ry_circuit)

0: ───H───Ry(0.5π)───


In [ ]:
llm_ry = """import cirq\nimport cirq\n\n# Define the circuit\nqubit = cirq.GridQubit(0, 0)\ncircuit = cirq.Circuit(\n    cirq.H(qubit),\n    cirq.measure(qubit, key=\'z\')\n)\n\n# Define the initial state\nstate = [0, 1]  # 0: |0⟩, 1: |1⟩\n\n\n# Apply the circuit to the state\nresult = cirq.(affine_triangular()).simulate(circuit, qubit).histogram(key=\'z\')\nprint(result.histogram(key="z"))\n\n# Output: \n# Histogram({\'0\': 8388607, \'1\': 8388607, \'\': 0})\n\n# This is close to the expected output, which would be:\n# Histogram({\'0\': 0, \'1\': 8388607, \'\': 0})\n# but due to numerical inaccuracies in the simulator, the probability of 0 is very small\n"""

validate_circuit(llm_ry, ry_circuit)

{'syntax': {'passed': False, 'error': 'invalid syntax (<string>, line 16)'},
 'verdict': 'invalid_code'}

In [ ]:
llm_amb = """python\nimport cirq\nimport numpy as np\n\n# Define two qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\n\n# Create a circuit with a Hadamard gate on each qubit to prepare an entangled state\ncircuit = cirq.Circuit(\n    cirq.H(q0),\n    cirq.H(q1),\n    c1')\n)\n\n# Create a Simulator and run the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(cirq.measure(q0, key='q0'),\n    cirq.measure(q1, key='qircuit, repetitions=1000)\n\n# Print the results\nprint(result.histogram(key='q0'))\nprint(result.histogram(key='q1'))\n"""

validate_circuit(llm_amb, bell_circuit)

0: ───H───M('q0')───

1: ───H───M('q1')───
0: ───H───M('q0')───

1: ───H───M('q1')───


KeyError: 'result'

In [ ]:
theta = np.pi / 2
rz_circuit = cirq.Circuit(cirq.rz(theta)(cirq.LineQubit(0)))
print(rz_circuit)

0: ───Rz(0.5π)───


In [ ]:
llm_rz = """import cirq\n\n# Define a qubit\nqubit = cirq.LineQubit(0)\n\n# Define the rotation angle\ntheta = cirq.pi / 2\n\n# Create a circuit that rotates the qubit around the Z axis by theta\ncircuit = cirq.Circuit(cirq.Rz(theta).on(qubit))\n\n# Print the circuit\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nprint("Final State:")\nprint(result.state)\n"""

validate_circuit(llm_rz, rz_circuit)

{'syntax': {'passed': False, 'error': "module 'cirq' has no attribute 'pi'"},
 'verdict': 'invalid_code'}

In [ ]:
llm_x = """import cirq\n\n# Create a Circuit with one qubit\ncircuit = cirq.Circuit(\ncirq.X(cirq.LineQubit(0))\n)\n\n# Print the Circuit\nprint(circuit)\n"""

validate_circuit(llm_x, x_circuit)

0: ───X───
0: ───X───


{'syntax': {'passed': True, 'error': None},
 'structure': {'qubit_count_match': True,
  'gate_types_match': True,
  'llm_gates': ['_PauliX'],
  'control_gates': ['_PauliX'],
  'measurement_match': True,
  'circuit_depth_llm': 1,
  'circuit_depth_control': 1,
  'depth_within_tolerance': True},
 'semantics': {'passed': True,
  'fidelity_score': 1.0,
  'llm_distribution': {'(1,)': 1.0},
  'control_distribution': {'(1,)': 1.0}},
 'verdict': 'success'}

In [ ]:
x_circuit = cirq.Circuit()
q0 = cirq.LineQubit(0)
x_circuit += cirq.X(q0)
print(x_circuit)

0: ───X───


In [ ]:
hm_circuit = cirq.Circuit()
q0 = cirq.LineQubit(0)
hm_circuit += cirq.H(q0)
hm_circuit += cirq.measure(q0, key='result')
print(hm_circuit)

0: ───H───M('result')───


In [ ]:
llm_hm = """python\nimport cirq\n\n# Specify the simulator to use\nsimulator = cirq.Simulator()\n\n# Define the qubit to use\nqubit = cirq.LineQubit(0)\n\n# Create a circuit and add a Hadamard gate to make the qubit's state 50/50 |1> and |0>\ncircuit = cirq.Circuit(\n    cirq.H(qubit)  # Hadamard gate to make the qubit's state 50/50 |1> and |0>\n)\n\n# Add a measurement gate to the circuit\ncircuit.append(cirq.measure(qubit, key='result'))\n\n# Run the circuit\nresult = simulator.run(circuit, repetitions=1000)\n\n# Print the measurement results\nprint(result.histogram(key='result', bins=2))\n"""

validate_circuit(llm_hm, hm_circuit)

0: ───H───M('result')───
0: ───H───M('result')───


{'syntax': {'passed': True, 'error': None},
 'structure': {'qubit_count_match': True,
  'gate_types_match': True,
  'llm_gates': ['MeasurementGate', 'HPowGate'],
  'control_gates': ['MeasurementGate', 'HPowGate'],
  'measurement_match': True,
  'circuit_depth_llm': 2,
  'circuit_depth_control': 2,
  'depth_within_tolerance': True},
 'semantics': {'passed': True,
  'fidelity_score': 0.9999998807907104,
  'llm_distribution': {'(0,)': 0.513, '(1,)': 0.487},
  'control_distribution': {'(0,)': 0.492, '(1,)': 0.508}},
 'verdict': 'success'}

In [ ]:
bellm_circuit = cirq.Circuit()
q0,q1 = cirq.LineQubit.range(2)
bellm_circuit += cirq.H(q0)
bellm_circuit += cirq.CNOT(q0,q1)
bellm_circuit += cirq.measure(q0, key='q0')
bellm_circuit += cirq.measure(q1, key='q1')
print(bellm_circuit)

0: ───H───@───M('q0')───
          │
1: ───────X───M('q1')───


In [ ]:
llm_bellm = """import cirq\nimport numpy as np\n\n# Define two qubits\nq0 = cirq.GridQubit(0, 0)\nq1 = cirq.GridQubit(0, 1)\n\n# Create a circuit with Hadamard gate on the first qubit and CNOT gate to produce 00 and 11 with equal probablility,\n# then append identity with pair of qubits to preserve superposition length in both the states 00 and 11\ncircuit = cirq.Circuit(\n    cirq.H(q0),\n    cirq.CNOT(q0, q1),\n    cirq.CX(q0,q1)\n)\n\n# Perform the circuit and measure both qubits\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=10000)\n\n# Print the results\nfor outcome in cirq.measure([q0, q1], key=\'outcome\'):\n    print(f"qubits are: {outcome}")\nprint("The probability of getting 00 is: ", result.histogram(key=\'outcome\')[0]/10000)\nprint("The probability of getting 11 is: ", result.histogram(key=\'outcome\')[1]/10000)\n"""

validate_circuit(llm_bellm,bellm_circuit)

{'syntax': {'passed': False,
  'error': 'Circuit has no measurements to sample.'},
 'verdict': 'invalid_code'}

In [ ]:
z_circuit = cirq.Circuit()
q0 = cirq.LineQubit(0)
z_circuit += cirq.Z(q0)
print(z_circuit)
print(cirq.final_state_vector(z_circuit))


0: ───Z───
[ 1.+0.j -0.+0.j]


In [ ]:
llm_z = """import cirq\n\n# Define the qubits\nq = cirq.LineQubit(0)\n\n# Define the phase flip operation\nphase_flip = cirq.X(q)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(phase_flip, repetitions=1)\n\n# Print the result\nprint(result)\n"""
validate_circuit(llm_z,z_circuit)

{'syntax': {'passed': False,
  'error': "'SingleQubitPauliStringGateOperation' object has no attribute 'has_measurements'"},
 'verdict': 'invalid_code'}

In [ ]:
cnot_circuit = cirq.Circuit()
q0,q1 = cirq.LineQubit.range(2)
cnot_circuit += cirq.CNOT(q0,q1)
cnot_circuit

0: ───@───
      │
1: ───X───

In [ ]:
llm_h = """import cirq\n\n# Create two qubits\nq0 = cirq.LineQubit(0)\n\n# Create a circuit containing an H gate applied to q0\ncircuit = cirq.Circuit(\n    cirq.H(q0),  # Apply Hadamard gate to q0\n)\n\nprint(circuit)  # Print the circuit\n"""

validate_circuit(llm_h, h_circuit)

0: ───H───
0: ───H───


{'syntax': {'passed': True, 'error': None},
 'structure': {'qubit_count_match': True,
  'gate_types_match': True,
  'llm_gates': ['HPowGate'],
  'control_gates': ['HPowGate'],
  'measurement_match': True,
  'circuit_depth_llm': 1,
  'circuit_depth_control': 1,
  'depth_within_tolerance': True},
 'semantics': {'passed': True,
  'fidelity_score': 0.9999998807907104,
  'llm_distribution': {'(0,)': 0.487, '(1,)': 0.513},
  'control_distribution': {'(0,)': 0.462, '(1,)': 0.538}},
 'verdict': 'success'}

In [ ]:
import json
[{
"id": "bell_technical_001",
"prompt": "Prepare the Bell state |Phi+> on two qubits as a Cirq circuit",
"prompt_style":"technical",
"category": "bell_state",
"difficulty": "easy",
"num_qubits": 2,
"expected_gates": ["H", "CNOT"],
"expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability."
"resulted code": "```python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit(\n    # Apply Hadamard to qubit0\n    cirq.H(qubit0),\n    # Apply CNOT with qubit0 as control and qubit1 as target\n    cirq.CNOT(qubit0, qubit1)\n)\n\n# Print the circuit\nprint("Circuit to prepare |Φ⁺⟩:")\nprint(circuit)\n```",
"validated verdict": "success"
},
{
"id": "bell_conversational_001",
"prompt": "Entangle 2 qubits as a Cirq Circuit",
"prompt_style":"conversational",
"category": "bell_state",
"difficulty": "easy",
"num_qubits": 2,
"expected_gates": ["H", "CNOT"],
"expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability.",
"resulted_code": "python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n# Add a Hadamard gate to qubit0 to create superposition\ncircuit.append(cirq.H(qubit0))\n\n# Add a CNOT gate to entangle qubit0 and qubit1\ncircuit.append(cirq.CNOT(qubit0, qubit1))\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nprint("\\nFinal state:)\nprint(result.final_state)\n",
"validated_verdict:": "invalid_code, inccorectly structured print statement, hallucinated state vector code",
},
{
"id": "bell_narrative_001",
"prompt": "Generate a quantum circuit that produces 00 and 11 with equal probibility",
"prompt_style":"narrative",
"category": "bell_state",
"difficulty": "easy",
"num_qubits": 2,
"expected_gates": ["H", "CNOT"],
"expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability.",
"resulted_code": "python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit()\ncircuit.append(cirq.H(qubit0))  # Apply Hadamard to qubit0\ncircuit.append(cirq.CNOT(qubit0, qubit1))  # Apply CNOT with qubit0 as control and qubit1 as target\n\n# Print the circuit\nprint("Circuit:")\n print(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nfinal_state = result.final_state_vector\n\nprint("\\nFinal state vector:")\nprint(final_state)\n\n# Measure the probabilities\nprint("\\nMeasurement probabilities:")\nprint(result.measurements)\n",
"validated_verdict": "invalid code, spacing error", 

}
{
"id": "h_technical_001",
"prompt": "Apply a Hadamard gate to a single qubit initialized in the |0⟩ computational basis state and return the resulting Cirq circuit.",
"prompt_style":"technical",
"category": "h_state",
"difficulty": "easy",
"num_qubits": 1,
"expected_gates": ["H"],
"expected_behavior": "After H is applied the qubit is put into superposition"
"resulted code:```python\nimport cirq\n\n# Create a single qubit circuit\nqubit = cirq.LineQubit(0)\ncircuit = cirq.Circuit()\n\n# Add the Hadamard gate to the qubit\ncircuit.append(cirq.H(qubit))\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n```",
"validated verdict: success"
""
},
{
"id": "h_conversational_001",
"prompt": "Put a qubit in superposition and model as a Cirq Circuit",
"prompt_style":"conversational",
"category": "h_state",
"difficulty": "easy",
"num_qubits": 1,
"expected_gates": ["H"],
"expected_behavior": "After H is applied the qubit is put into superposition",
"resulted_code": "python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n# Add a Hadamard gate to qubit0 to create superposition\ncircuit.append(cirq.H(qubit0))\n\n# Add a CNOT gate to entangle qubit0 and qubit1\ncircuit.append(cirq.CNOT(qubit0, qubit1))\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nprint("\\nFinal state:)\nprint(result.final_state)\n",
"validated_verdict:": "invalid_code",
},
{
"id": "h_narrative_001",
"prompt": "Generate a Cirq quantum circuit that produces 0 and 1 with equal probibility",
"prompt_style":"narrative",
"category": "h_state",
"difficulty": "easy",
"num_qubits": 1,
"expected_gates": ["H"],
"expected_behavior": "After H is applied the qubit is put into superposition",
"resulted_code": "```import cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit()\ncircuit.append(cirq.H(qubit0))  # Apply Hadamard to qubit0\ncircuit.append(cirq.CNOT(qubit0, qubit1))  # Apply CNOT with qubit0 as control and qubit1 as target\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nfinal_state = result.final_state_vector\n\nprint("\\nFinal state vector:")\nprint(final_state)\n\n# Measure the probabilities\nprint("\\nMeasurement probabilities:")\nprint(result.measurements)\n```",
"validated_verdict": "success, but incorrect mesurement use", }
{
"id": "tele_conversational_001",
"prompt": "Generate a cirq circuit modeling quantum teleportation.",
"prompt_style":"conversational",
"category": "teleportation",
"difficulty": "easy",
"num_qubits": 3,
"expected_gates": ["H, CNOT, CNOT, H, M, X, Z"],
"expected_behavior": "After H is applied the qubit is put into superposition",
"resulted_code": "```import cirq\n\n# Create a qubit for the state to be teleported\nqubit_for_teleportation = cirq.LineQubit(0)\n\n# Create a qubit for the entangled pair\nqubit_entangled_pair = cirq.LineQubit(1)\n\n# Create a qubit for the receiver\nqubit_receiver = cirq.LineQubit(2)\n\n# Create the entangled pair (EPR pair)\nepr_pair = cirq.H(qubit_entangled_pair)\nepr_pair_with_control = cirq.X(qubit_entangled_pair) ** cirq.ControlledBy(cirq.X(qubit_receiver))\n\n# Prepare the state to be teleported\nstate_to_teleport = cirq.X(qubit_for_teleportation)\n\n# Apply the teleportation circuit\nteleportation_circuit = cirq.Circuit(\n    epr_pair,\n    cirq.CNOT(qubit_for_teleportation, qubit_entangled_pair),\n    cirq.H(qubit_for_teleportation),\n    cirq.measure(qubit_for_teleportation, key='teleported_state'),\n    cirq.measure(qubit_entangled_pair, key='entangled_pair'),\n    cirq.measure(qubit_receiver, key='receiver_state')\n)\n\n# Apply the measurement on the receiver's qubit\nreceiver_measurement = cirq.measure(qubit_receiver, key='receiver_measurement')\n\n# Apply the correction to the receiver's qubit\ncorrection = cirq.X(qubit_receiver) ** cirq.ControlledBy(cirq.X(qubit_entangled_pair))\n\n# Combine all the operations into a single circuit\nteleportation_circuit = cirq.Circuit(\n    teleportation_circuit,\n    receiver_measurement,\n    correction\n)\n\n# Run the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(teleportation_circuit, repetitions=1000)\n\n# Print the results\nprint(result)\n```",
"validated_verdict": "invalid code, halluciogated code, right idea though", }
{
"id": "tele_narrative_001",
"prompt": "Generate a cirq circuit that transfers the quantum state of the message qubit to the receiver qubit using only entanglement and classical communication. After the protocol, the receiver qubit's state should be identical (up to global phase) to the original message qubit's state. The message qubit is destroyed in the process — this is not cloning.",
"prompt_style":"conversational",
"category": "teleportation",
"difficulty": "easy",
"num_qubits": 3,
"expected_gates": ["H, CNOT, CNOT, H, M, X, Z"],
"expected_behavior": "After H is applied the qubit is put into superposition",
"resulted_code": "```import cirq\n\n# Create a qubit for the state to be teleported\nqubit_for_teleportation = cirq.LineQubit(0)\n\n# Create a qubit for the entangled pair\nqubit_entangled_pair = cirq.LineQubit(1)\n\n# Create a qubit for the receiver\nqubit_receiver = cirq.LineQubit(2)\n\n# Create the entangled pair (EPR pair)\nepr_pair = cirq.H(qubit_entangled_pair)\nepr_pair_with_control = cirq.X(qubit_entangled_pair) ** cirq.ControlledBy(cirq.X(qubit_receiver))\n\n# Prepare the state to be teleported\nstate_to_teleport = cirq.X(qubit_for_teleportation)\n\n# Apply the teleportation circuit\nteleportation_circuit = cirq.Circuit(\n    epr_pair,\n    cirq.CNOT(qubit_for_teleportation, qubit_entangled_pair),\n    cirq.H(qubit_for_teleportation),\n    cirq.measure(qubit_for_teleportation, key='teleported_state'),\n    cirq.measure(qubit_entangled_pair, key='entangled_pair'),\n    cirq.measure(qubit_receiver, key='receiver_state')\n)\n\n# Apply the measurement on the receiver's qubit\nreceiver_measurement = cirq.measure(qubit_receiver, key='receiver_measurement')\n\n# Apply the correction to the receiver's qubit\ncorrection = cirq.X(qubit_receiver) ** cirq.ControlledBy(cirq.X(qubit_entangled_pair))\n\n# Combine all the operations into a single circuit\nteleportation_circuit = cirq.Circuit(\n    teleportation_circuit,\n    receiver_measurement,\n    correction\n)\n\n# Run the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(teleportation_circuit, repetitions=1000)\n\n# Print the results\nprint(result)\n```",
"validated_verdict": "invalid code, halluciogated code, right idea though", }
{
"id": "fourier_conversational_001",
"prompt": "Generate a cirq circuit modeling Quantum Fourier Transform with 3 qubits.",
"prompt_style":"conversational",
"category": "teleportation",
"difficulty": "easy",
"num_qubits": 3,
"expected_gates": ["H, CPHASE, SWAP"],
"expected_behavior": "After H is applied the qubit is put into superposition",
"resulted_code": "```python\nimport cirq\n\n# Define the qubits\nq0 = cirq.GridQubit(0, 0)\nq1 = cirq.GridQubit(0, 1)\nq2 = cirq.GridQubit(0, 2)\n\n# Define the angles for controlled rotations in QFT\ndef qft_rotation(angles, num_bits):\n    for i in range(num_bits):\n        for j in range(i + 1, num_bits):\n            factor = 2 ** (j - i - 1)\n            cir = cirq.Circuit()\n            cir.append(cirq.H(qubits=[q0, q1, q2][i]))\n            cir.append(cirq_RCCQouch_Xdot(circuit=cirq.Circuit(), qubit=cirq.GridQubit(0, 0)))\n            cir.append(cirq.XPowGate(exponent=factor * angles).on(cirq.GridQubit(0, 0)))\n            yield cir\n\n# Define the QFT circuit\nnum_bits = 3\ncirc = cirq.Circuit()\nfor hist in qft_rotation(angles=[1.5707963267948966, 1.5707963267948966, 1.5707963267948966], num_bits=3):\n    circ.append(hist)\n\nprint(circ)\n```",
"validated_verdict": "invalid code, halluciogated code, silly total output, understood the math, right idea", }
{
"id": "grover_conversational_001",
"prompt": "Generate a cirq circuit modeling Grover's Diffusion Operator on 3 qubits.",
"prompt_style":"conversational",
"category": "grover's diffusion",
"difficulty": "easy",
"num_qubits": 3,
"expected_gates": ["H, X, H, CNOT, H, X, H"],
"expected_behavior": "After H is applied the qubit is put into superposition",
"resulted_code": "```import cirq\nimport math\n\n# Define a function to create the diffusion operator on a single qubit\ndef diffusion_operator(qubit):\n    return cirq.H(qubit) * (\n        cirq.X powder cirq.X(qubit)) * cirq.H(qubit)\n\n# Create a circuit and add the diffusion operator to each qubit\ncircuit = cirq.Circuit()\nfor qubit in ['q0', 'q1', 'q2']:\n    circuit.append(diffusion_operator(cirq.LineQubit(int(qubit))))\n    circuit.append(cirq.measure(cirq.LineQubit(int(qubit)), key=qubit))\n\n# Set up the simulator and execute the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\nprint(result)\n```",
"validated_verdict": "invalid code, inproper syntax, halluciogated code, powder, multiplying operations instead of sequence, less right idea", }
{
"id": "toffoli_conversational_001",
"prompt": "Generate a cirq circuit modeling a toffoli gate.",
"prompt_style":"conversational",
"category": "toffoli",
"difficulty": "easy",
"num_qubits": 3,
"expected_gates": ["CCNOT or CCX"],
"expected_behavior": "After H is applied the qubit is put into superposition",
"resulted_code": "import cirq\n\n# Define the three qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\nq2 = cirq.LineQubit(2)\n\n# Define the Toffoli gate operation\ntoffoli_gate = cirq.Circuit(\n    cirq.X(q0) ** cirq.LineairX gateughter(1)**(1, 0),\n    cirq.CNOT(q1, q2),\n    cirq.X(q0) ** cirq.LineairX gateughter(1)**(1, 0),\n)\n\n# Alternatively using standard cirq Toffoli gate\ntoffoli_gate_std = cirq.Toffoli(q0, q1, q2)\n\n# To execute the circuit\nsimulator = cirq.Simulator()\nprint(simulator.simulate(cirqxford(\'H\' ** cirq.HourTooltip Dop允 positioning Ur())))\n",
"validated_verdict": "invalid code, a lot of halucinations, corrupted characters, "}
{
"id": "kickback_conversational_001",
"prompt": "Generate a cirq circuit modeling a phase kickback.",
"prompt_style":"conversational",
"category": "phase kick back",
"difficulty": "easy",
"num_qubits": 2,
"expected_gates": ["H, CNOT"],
"expected_behavior": "After H is applied the qubit is put into superposition",
"resulted_code": "import cirq\n\n# Create two qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\n\n# Apply the CNOT gate to q0 and q1\ncircuit = cirq.Circuit(\n    cirq.X(q0) ** 0.5,  # Optional, applies a Hadamard gate to q0\n    cirq.CNOT(q0, q1),\n    cirq.MCTLayers()[0][cirq.MeasurementGate(1)()(q0)],  # Measure q0\n    cirq.measure(q0, key='q0'),\n    cirq.measure(q1, key='q1')\n)\n\n# Simulate the circuit and print the output\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\n\nprint(result.histogram(key='q0'))\n",
"validated_verdict": "invalid code, a lot of halucinations, inproper gates, "}
{
"id": "twovari_conversational_001",
"prompt": "Generate a Two-Layer Variational Circuit as a cirq circuit.",
"prompt_style":"conversational",
"category": "phase kick back",
"difficulty": "easy",
"num_qubits": 2,
"expected_gates": ["H, CNOT"],
"expected_behavior": "After H is applied the qubit is put into superposition",
"resulted_code": "import cirq\n\n# Create two qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\n\n# Apply the CNOT gate to q0 and q1\ncircuit = cirq.Circuit(\n    cirq.X(q0) ** 0.5,  # Optional, applies a Hadamard gate to q0\n    cirq.CNOT(q0, q1),\n    cirq.MCTLayers()[0][cirq.MeasurementGate(1)()(q0)],  # Measure q0\n    cirq.measure(q0, key='q0'),\n    cirq.measure(q1, key='q1')\n)\n\n# Simulate the circuit and print the output\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\n\nprint(result.histogram(key='q0'))\n",
"validated_verdict": "invalid code, a lot of halucinations, inproper gates, "}
]


SyntaxError: invalid character '⁺' (U+207A) (3765424421.py, line 11)

In [37]:
# Two-layer variational circuit (3 qubits)
qubits = cirq.LineQubit.range(3)

variational_circuit = cirq.Circuit(
    # Layer 1: parameterized single-qubit rotations
    [cirq.ry(0.6)(q) for q in qubits],
    # Entangling layer
    [cirq.CNOT(qubits[i], qubits[i+1]) for i in range(len(qubits)-1)],
    # Layer 2: more rotations
    [cirq.ry(1.2)(q) for q in qubits],
    # Second entangling layer
    [cirq.CNOT(qubits[i], qubits[i+1]) for i in range(len(qubits)-1)],
    # Measurements
    cirq.measure(*qubits, key='m')
)

print(variational_circuit)

0: ───Ry(0.191π)───@───Ry(0.382π)────────────────@───────M('m')───
                   │                             │       │
1: ───Ry(0.191π)───X───@────────────Ry(0.382π)───X───@───M────────
                       │                             │   │
2: ───Ry(0.191π)───────X────────────Ry(0.382π)───────X───M────────


In [ ]:
q

In [36]:
kickback_circuit = cirq.Circuit()

q0,q1 = cirq.LineQubit.range(2)

kickback_circuit.append(cirq.H(q0))
kickback_circuit += cirq.CNOT(q0, q1)
kickback_circuit.append(cirq.H(q0))
print(kickback_circuit)


0: ───H───@───H───
          │
1: ───────X───────


In [ ]:
kickback_llm = """import cirq\n\n# Create two qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\n\n# Apply the CNOT gate to q0 and q1\ncircuit = cirq.Circuit(\n    cirq.X(q0) ** 0.5,  # Optional, applies a Hadamard gate to q0\n    cirq.CNOT(q0, q1),\n    cirq.MCTLayers()[0][cirq.MeasurementGate(1)()(q0)],  # Measure q0\n    cirq.measure(q0, key='q0'),\n    cirq.measure(q1, key='q1')\n)\n\n# Simulate the circuit and print the output\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\n\nprint(result.histogram(key='q0'))\n"""
validate_circuit(kickback_llm, kickback_circuit)

{'syntax': {'passed': False,
  'error': "module 'cirq' has no attribute 'MCTLayers'"},
 'verdict': 'invalid_code'}

In [ ]:
toffoli_circuit = cirq.Circuit()

q0,q1,q2 = cirq.LineQubit.range(3) 
toffoli_circuit.append(cirq.CCNOT(q0,q1,q2))
print(toffoli_circuit)

0: ───@───
      │
1: ───@───
      │
2: ───X───


In [ ]:
toffoli_llm = "import cirq\n\n# Define the three qubits\nq0 = cirq.LineQubit(0)\nq1 = cirq.LineQubit(1)\nq2 = cirq.LineQubit(2)\n\n# Define the Toffoli gate operation\ntoffoli_gate = cirq.Circuit(\n    cirq.X(q0) ** cirq.LineairX gateughter(1)**(1, 0),\n    cirq.CNOT(q1, q2),\n    cirq.X(q0) ** cirq.LineairX gateughter(1)**(1, 0),\n)\n\n# Alternatively using standard cirq Toffoli gate\ntoffoli_gate_std = cirq.Toffoli(q0, q1, q2)\n\n# To execute the circuit\nsimulator = cirq.Simulator()\nprint(simulator.simulate(cirqxford(\'H\' ** cirq.HourTooltip Dop允 positioning Ur())))\n"

validate_circuit(toffoli_llm, toffoli_circuit)


{'syntax': {'passed': False,
  'error': 'invalid syntax. Perhaps you forgot a comma? (<string>, line 10)'},
 'verdict': 'invalid_code'}

In [32]:
grover_circuit = cirq.Circuit()

q0,q1,q2 = cirq.LineQubit.range(3) 
grover_circuit.append([cirq.H(q0), cirq.H(q1), cirq.H(q2)])
grover_circuit.append([cirq.X(q0), cirq.X(q1), cirq.X(q2)])
grover_circuit += cirq.CCZ(q0,q1,q2)
grover_circuit.append([cirq.X(q0), cirq.X(q1), cirq.X(q2)])
grover_circuit.append([cirq.H(q0), cirq.H(q1), cirq.H(q2)])
print(grover_circuit)


0: ───H───X───@───X───H───
              │
1: ───H───X───@───X───H───
              │
2: ───H───X───@───X───H───


In [33]:
grov_llm = """import cirq\nimport math\n\n# Define a function to create the diffusion operator on a single qubit\ndef diffusion_operator(qubit):\n    return cirq.H(qubit) * (\n        cirq.X powder cirq.X(qubit)) * cirq.H(qubit)\n\n# Create a circuit and add the diffusion operator to each qubit\ncircuit = cirq.Circuit()\nfor qubit in ['q0', 'q1', 'q2']:\n    circuit.append(diffusion_operator(cirq.LineQubit(int(qubit))))\n    circuit.append(cirq.measure(cirq.LineQubit(int(qubit)), key=qubit))\n\n# Set up the simulator and execute the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(circuit, repetitions=1000)\nprint(result)\n"""

validate_circuit(grov_llm, grover_circuit)

{'syntax': {'passed': False,
  'error': 'invalid syntax. Perhaps you forgot a comma? (<string>, line 7)'},
 'verdict': 'invalid_code'}

In [34]:
fourier_circuit = cirq.Circuit()

q0,q1,q2 = cirq.LineQubit.range(3)
fourier_circuit += cirq.H(q0)
fourier_circuit += cirq.CZ(q0, q1)
fourier_circuit += cirq.T(q0)
fourier_circuit += cirq.H(q1)
fourier_circuit += cirq.CZ(q1, q2)
fourier_circuit += cirq.H(q2)
fourier_circuit += cirq.SWAP(q2,q0)
print(fourier_circuit)

0: ───H───@───T───────────×───
          │               │
1: ───────@───H───@───────┼───
                  │       │
2: ───────────────@───H───×───


In [ ]:
four_llm = """import cirq\n\n# Define the qubits\nq0 = cirq.GridQubit(0, 0)\nq1 = cirq.GridQubit(0, 1)\nq2 = cirq.GridQubit(0, 2)\n\n# Define the angles for controlled rotations in QFT\ndef qft_rotation(angles, num_bits):\n    for i in range(num_bits):\n        for j in range(i + 1, num_bits):\n            factor = 2 ** (j - i - 1)\n            cir = cirq.Circuit()\n            cir.append(cirq.H(qubits=[q0, q1, q2][i]))\n            cir.append(cirq_RCCQouch_Xdot(circuit=cirq.Circuit(), qubit=cirq.GridQubit(0, 0)))\n            cir.append(cirq.XPowGate(exponent=factor * angles).on(cirq.GridQubit(0, 0)))\n            yield cir\n\n# Define the QFT circuit\nnum_bits = 3\ncirc = cirq.Circuit()\nfor hist in qft_rotation(angles=[1.5707963267948966, 1.5707963267948966, 1.5707963267948966], num_bits=3):\n    circ.append(hist)\n\nprint(circ)\n"""
validate_circuit(four_llm, fourier_circuit)

{'syntax': {'passed': False,
  'error': 'Wrong number of qubits for <cirq.H>. Expected 1 qubits but got <()>.'},
 'verdict': 'invalid_code'}

In [35]:
teleportion_circuit = cirq.Circuit()

q0, q1,q2 = cirq.LineQubit.range(3)
teleportion_circuit += cirq.H(q2)
teleportion_circuit += cirq.CX(q2, q1)
teleportion_circuit += cirq.CX(q0, q1)
teleportion_circuit += cirq.H(q0)
teleportion_circuit += cirq.measure(q0, key='q0')
teleportion_circuit += cirq.measure(q1, key='q1')
teleportion_circuit += cirq.Z(q2).with_classical_controls('q0')
teleportion_circuit += cirq.X(q2).with_classical_controls('q1')
print(teleportion_circuit)

0: ────────────@───H───M───────────
               │       ║
1: ────────X───X───M───╫───────────
           │       ║   ║
2: ────H───@───────╫───╫───Z───X───
                   ║   ║   ║   ║
q0: ═══════════════╬═══@═══^═══╬═══
                   ║           ║
q1: ═══════════════@═══════════^═══


In [ ]:
teleportation_test = """import cirq\n\n# Create a qubit for the state to be teleported\nqubit_for_teleportation = cirq.LineQubit(0)\n\n# Create a qubit for the entangled pair\nqubit_entangled_pair = cirq.LineQubit(1)\n\n# Create a qubit for the receiver\nqubit_receiver = cirq.LineQubit(2)\n\n# Create the entangled pair (EPR pair)\nepr_pair = cirq.H(qubit_entangled_pair)\nepr_pair_with_control = cirq.X(qubit_entangled_pair) ** cirq.ControlledBy(cirq.X(qubit_receiver))\n\n# Prepare the state to be teleported\nstate_to_teleport = cirq.X(qubit_for_teleportation)\n\n# Apply the teleportation circuit\nteleportation_circuit = cirq.Circuit(\n    epr_pair,\n    cirq.CNOT(qubit_for_teleportation, qubit_entangled_pair),\n    cirq.H(qubit_for_teleportation),\n    cirq.measure(qubit_for_teleportation, key='teleported_state'),\n    cirq.measure(qubit_entangled_pair, key='entangled_pair'),\n    cirq.measure(qubit_receiver, key='receiver_state')\n)\n\n# Apply the measurement on the receiver's qubit\nreceiver_measurement = cirq.measure(qubit_receiver, key='receiver_measurement')\n\n# Apply the correction to the receiver's qubit\ncorrection = cirq.X(qubit_receiver) ** cirq.ControlledBy(cirq.X(qubit_entangled_pair))\n\n# Combine all the operations into a single circuit\nteleportation_circuit = cirq.Circuit(\n    teleportation_circuit,\n    receiver_measurement,\n    correction\n)\n\n# Run the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(teleportation_circuit, repetitions=1000)\n\n# Print the results\nprint(result)\n"""
validate_circuit(teleportation_test, teleportion_circuit)

{'syntax': {'passed': False,
  'error': "module 'cirq' has no attribute 'ControlledBy'"},
 'verdict': 'invalid_code'}

In [ ]:
test_circuit = """import cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n# Add a Hadamard gate to qubit0 to create superposition\ncircuit.append(cirq.H(qubit0))\n\n# Add a CNOT gate to entangle qubit0 and qubit1\ncircuit.append(cirq.CNOT(qubit0, qubit1))\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nprint("\\nFinal state:)\nprint(result.final_state)\n"""

validate_circuit(test_circuit, h_circuit)

{'syntax': {'passed': False,
  'error': 'unterminated string literal (detected at line 23) (<string>, line 23)'},
 'verdict': 'invalid_code'}

In [ ]:
bt_code = """import cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit()\ncircuit.append(cirq.H(qubit0))  # Apply Hadamard to qubit0\ncircuit.append(cirq.CNOT(qubit0, qubit1))  # Apply CNOT with qubit0 as control and qubit1 as target\n\n# Print the circuit\nprint("Circuit:")\n print(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nfinal_state = result.final_state_vector\n\nprint("\\nFinal state vector:")\nprint(final_state)\n\n# Measure the probabilities\nprint("\\nMeasurement probabilities:")\nprint(result.measurements)\n"""
bell_test1_code = """import cirq

# Define the qubits
qubit0 = cirq.LineQubit(0)
qubit1 = cirq.LineQubit(1)

# Create the circuit
circuit = cirq.Circuit()
circuit.append(cirq.H(qubit0))  # Apply Hadamard to qubit0
circuit.append(cirq.CNOT(qubit0, qubit1))  # Apply CNOT with qubit0 as control and qubit1 as target

# Print the circuit
print("Circuit:")
print(circuit)

# Simulate the circuit
simulator = cirq.Simulator()
result = simulator.simulate(circuit)
final_state = result.final_state_vector

print("\\nFinal state vector:")
print(final_state)

# Measure the probabilities
print("\\nMeasurement probabilities:")
print(result.measurements)
"""
validate_circuit(bt_code, bell_circuit)


{'syntax': {'passed': False, 'error': 'unexpected indent (<string>, line 14)'},
 'verdict': 'invalid_code'}

In [ ]:
bell_circuit8 = """import cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit()\ncircuit.append(cirq.H(qubit0))  # Apply Hadamard to qubit0\ncircuit.append(cirq.CNOT(qubit0, qubit1))  # Apply CNOT with qubit0 as control and qubit1 as target\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nfinal_state = result.final_state_vector\n\nprint("\\nFinal state vector:")\nprint(final_state)\n\n# Measure the probabilities\nprint("\\nMeasurement probabilities:")\nprint(result.measurements)\n\n"""
validate_circuit(bell_circuit8, bell_circuit)

Circuit:
0: ───H───@───
          │
1: ───────X───

Final state vector:
[0.70710677+0.j 0.        +0.j 0.        +0.j 0.70710677+0.j]

Measurement probabilities:
{}
Circuit:
0: ───H───@───
          │
1: ───────X───

Final state vector:
[0.70710677+0.j 0.        +0.j 0.        +0.j 0.70710677+0.j]

Measurement probabilities:
{}


{'syntax': {'passed': True, 'error': None},
 'structure': {'qubit_count_match': True,
  'gate_types_match': True,
  'llm_gates': ['HPowGate', 'CXPowGate'],
  'control_gates': ['HPowGate', 'CXPowGate'],
  'measurement_match': True,
  'circuit_depth_llm': 2,
  'circuit_depth_control': 2,
  'depth_within_tolerance': True},
 'semantics': {'passed': True,
  'fidelity_score': 0.9999998807907104,
  'llm_distribution': {'(0,)': 0.477, '(3,)': 0.523},
  'control_distribution': {'(3,)': 0.484, '(0,)': 0.516}},
 'verdict': 'success'}